In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torchvision.ops import StochasticDepth

import torch.backends.cudnn as cudnn
cudnn.benchmark = True

In [2]:
class TinyImageNetDataset(Dataset):
    def __init__(self, root_dir, transform=None, train=True):
        self.root_dir = root_dir

        self.train = train
        self.transform = transform

        if self.train:
            self.data = []
            self.labels = []
            classes = sorted(os.listdir(os.path.join(root_dir, 'train')))

            # Поиск картинок и присвоение labels
            for label, cls in enumerate(classes):
                cls_dir = os.path.join(root_dir, 'train', cls, 'images')
                for img_name in os.listdir(cls_dir):
                    self.data.append(os.path.join(cls_dir, img_name)) # store the path only
                    self.labels.append(label)
        else:
            self.data = []
            self.labels = []
            val_dir = os.path.join(root_dir, 'val', 'images')

            # Чтение csv файлов с данными
            val_annotations = pd.read_csv(os.path.join(root_dir, 'val', 'val_annotations.txt'),
                                          sep='\t', header=None,
                                          names=['file_name', 'class', 'x1', 'y1', 'x2', 'y2'])
            class_to_idx = {cls: idx for idx, cls in enumerate(sorted(os.listdir(os.path.join(root_dir, 'train'))))}
            for _, row in val_annotations.iterrows():
                self.data.append(os.path.join(val_dir, row['file_name']))
                self.labels.append(class_to_idx[row['class']])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data[idx]
        # Загрузка изображения
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        # Применение трансформаций
        if self.transform:
            image = self.transform(image)

        return image, label

In [21]:
class DatasetCars(Dataset):
    def __init__(self, root, transform = None, train = None):
        self.root = root
        self.transform = transform
        self.train = train
        self.root = os.path.join(self.root, 'car_data')
        
        self.data = []
        self.labels = []
        
        if train == True:
            self.root = os.path.join(self.root, 'train')
            self.classes = os.listdir(self.root)
            self.class_to_idx = {}
            for label, cls in enumerate(self.classes):
                self.class_to_idx[label] = cls
                cls_dir = os.path.join(self.root, cls)
                for img in os.listdir(cls_dir):
                    self.data.append(os.path.join(cls, img))
                    self.labels.append(label)            
        else:
            self.root = os.path.join(self.root, 'test')
            self.classes = os.listdir(self.root)

            self.class_to_idx = {}
            for label, cls in enumerate(self.classes):
                self.class_to_idx[label] = cls
                cls_dir = os.path.join(self.root, cls)
                for img in os.listdir(cls_dir):
                    self.data.append(os.path.join(cls, img))
                    self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        #print(os.path.join(self.root), self.data[idx])
        image = Image.open(os.path.join(self.root, self.data[idx])).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, self.labels[idx]

In [15]:
class CustomDataset(Dataset):
    def __init__(self, root, transform = None):
        self.root = root
        self.transform = transform
        self.classes = os.listdir(root)
        self.classes_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.labels = []
        self.images = []
        for class_name in self.classes:
            class_dir = os.path.join(root, class_name)
            for image_name in os.listdir(class_dir):
                self.images.append(image_name)
                self.labels.append(self.classes_to_idx[class_name])
                
    def __len__(self):
        return len(self.images)
        
    def __getitem__(self, index):
        image_name = self.images[index]
        label = self.labels[index]
        label_name = self.classes[label]
        
        image_folder = os.path.join(self.root, label_name)
        image = Image.open(os.path.join(image_folder, image_name))
        if self.transform != None:
            image = self.transform(image)

        return image, label 


In [22]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomGrayscale(),
    #transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    #transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [67]:
train_root = r'C:\DATA\stanford-car-dataset-by-classes-folder-224'
val_root = train_root

train_data = DatasetCars(train_root, transform=train_transform, train=True)
train_loader = DataLoader(train_data, batch_size = 32, shuffle=True, num_workers=0)

val_data = DatasetCars(val_root, transform=val_transform, train=False)
val_loader = DataLoader(val_data, batch_size = 32, shuffle=False, num_workers=0)

In [75]:
train_data = TinyImageNetDataset(r'C:\DATA\tiny-imagenet-200', transform=train_transform, train=True)
val_data = TinyImageNetDataset(r'C:\DATA\tiny-imagenet-200', transform=val_transform, train=False)

BATCH_SIZE = 16
train_loader = DataLoader(train_data, batch_size= BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size= BATCH_SIZE, shuffle=False, num_workers=0)

In [7]:
#ResNet18

class BasicBlock(nn.Module):
    def __init__( self, in_channels, out_channels, identity_downsample = None, stride = 1):
        super(BasicBlock, self).__init__()
        self.layer = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.identity_downsample = identity_downsample
        
    def forward(self, X):
        identity = X
        X = self.layer(X)
        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)
        #print(X.shape, identity.shape)
        X += identity
        return F.relu(X)

class ResNet18(nn.Module):
    def __init__(self, img_channel = 3, num_classes = 6, block = BasicBlock(64, 64)):
        super(ResNet18, self).__init__()
        self.layer = nn.Sequential(
            nn.Conv2d(img_channel, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3, 2, 1)
        )

        self.layer1 = self.__make_layer(64, 64, 2, 1)
        self.layer2 = self.__make_layer(64, 128, 2, 2)
        self.layer3 = self.__make_layer(128, 256, 2, 2)
        self.layer4 = self.__make_layer(256, 512, 2, 2)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout2d(0.45)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def __make_layer(self, in_channels, out_channels, num_blocks, stride):
        identity_downsample = None
        
        if stride != 1:
            identity_downsample = self.identity_downsample(in_channels, out_channels)
        return nn.Sequential(
            BasicBlock(in_channels, out_channels, identity_downsample=identity_downsample, stride=stride), 
            BasicBlock(out_channels, out_channels))

    def identity_downsample(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 2, 1),
            nn.BatchNorm2d(out_channels))

    def forward(self, X):
        out = self.layer(X)
        
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avg_pool(out)
        out = self.dropout(out)
        out = self.flatten(out)
        out = self.fc(out)
        return out

In [8]:
#ResNet50

class BottleNeck(nn.Module):
    def __init__(self, in_channels, out_channels, stride, downsample = None, prob = 0):
        super(BottleNeck, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.drop_path = StochasticDepth(p = prob, mode = 'row')
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels = self.in_channels, out_channels = self.out_channels // 4, kernel_size = 1),
            nn.BatchNorm2d(out_channels // 4),
            nn.Conv2d(in_channels= self.out_channels // 4, out_channels = self.out_channels // 4, kernel_size = 3, stride = self.stride, padding = 1),
            nn.BatchNorm2d(out_channels // 4),
            nn.Conv2d(in_channels = self.out_channels // 4, out_channels = self.out_channels, kernel_size = 1),
            nn.BatchNorm2d(out_channels),
            self.drop_path
        )
        self.downsample = downsample
        self.relu = nn.ReLU()

    def forward(self, X):
        residual = X
        out = self.conv(X)
        if self.downsample is not None:
            residual = self.downsample(residual)
        out += residual
        return self.relu(out)
        
class ResNet50(nn.Module):
    def __init__(self, num_classes = 1000):
        super(ResNet50, self).__init__()
        self.conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride = 2, padding=3)
        self.bn = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2)
        self.layer1 = self.make_layer(64, 256, 3, 1)
        self.layer2 = self.make_layer(128, 512, 4, 2)
        self.layer3 = self.make_layer(256, 1024, 6, 2)
        self.layer4 = self.make_layer(512, 2048, 3, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Dropout2d(0.35),
            nn.Flatten(),
            nn.Linear(2048, num_classes)
        )
        
    def make_layer(self, in_channels, out_channels, blocks, stride):
        downsample = None
        layers = []
        if stride == 1:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1, stride=stride, bias = False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            )
            layers.append(BottleNeck(in_channels=in_channels, out_channels=out_channels, stride = stride, downsample=downsample, prob = 0.033))
        else:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels=in_channels * 2, out_channels=out_channels, kernel_size=1, stride=stride, bias = False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            )
            layers.append(BottleNeck(in_channels=in_channels * 2, out_channels=out_channels, stride = stride, downsample=downsample, prob = 0.033))
        in_channels = out_channels
        for i in range(1, blocks):
            layers.append(BottleNeck(in_channels, out_channels, 1, prob= (i + 1) * 0.033))
        return nn.Sequential(*layers)
            
    def forward(self, X):
        out = self.conv(X)
        out = self.bn(out)
        out = self.relu(out)
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = self.fc(out)
        return out

In [69]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
resnet50= ResNet50(num_classes = 196).to(device) #ResNet50(num_classes = 1000).to(device) 
loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(params=resnet50.parameters(), lr=learning_rate, weight_decay=1e-5)
epochs = 50

resnet50.load_state_dict(torch.load(f = 'resnet50', weights_only=True))

<All keys matched successfully>

In [68]:
%%time
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, eta_min = 1e-5, T_max=255, last_epoch=-1)
best_acc = 0

for epoch in range(epochs):   
    train_loop = tqdm(train_loader)
    train_loss = []
    resnet50.train()
    for X, y in train_loop:
        X = X.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        y_pred = resnet50(X)
        loss = loss_fn(y_pred, y)
        train_loss.append(loss)
        loss.backward()
        optimizer.step()
        scheduler.step()
    print(f'Epochs: {epoch + 1}/{epochs}, Loss: {(sum(train_loss) / len(train_loss))}, Learning Rate: {(scheduler.get_last_lr()[0])}')
    
    correct = 0
    resnet50.eval()
    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)
            y_pred = resnet50(X)
            pred = y_pred.argmax(dim = 1, keepdim = True)
            correct += pred.eq(y.view_as(pred)).sum().item()
    acc = np.round((100. * correct / len(val_data)), 4)
    if acc > best_acc:
        best_acc = acc
        torch.save(resnet50.state_dict(), f = 'resnet50')
    print(f'Accuracy: {acc}')

100%|████████████████████████████████████████████████████████████████████████████████| 255/255 [01:06<00:00,  3.85it/s]


Epochs: 1/50, Loss: 1.5890270471572876, Learning Rate: 1e-05
Accuracy: 41.8853


100%|████████████████████████████████████████████████████████████████████████████████| 255/255 [01:08<00:00,  3.72it/s]


Epochs: 2/50, Loss: 1.4031798839569092, Learning Rate: 0.01000000000000003
Accuracy: 7.4493


100%|████████████████████████████████████████████████████████████████████████████████| 255/255 [01:05<00:00,  3.91it/s]


...

Epochs: 48/50, Loss: 0.2558577060699463, Learning Rate: 0.010000000000029242
Accuracy: 23.6413


100%|████████████████████████████████████████████████████████████████████████████████| 255/255 [01:17<00:00,  3.30it/s]


Epochs: 49/50, Loss: 0.32904940843582153, Learning Rate: 1e-05
Accuracy: 46.8972


100%|████████████████████████████████████████████████████████████████████████████████| 255/255 [01:15<00:00,  3.36it/s]


Epochs: 50/50, Loss: 0.2749270498752594, Learning Rate: 0.009999999999970768
Accuracy: 13.8043
CPU times: total: 4h 56min 30s
Wall time: 1h 17min 32s


In [70]:
correct = 0
resnet50.eval()
with torch.no_grad():
    for X, y in val_loader:
        X = X.to(device)
        y = y.to(device)
        y_pred = resnet50(X)
        pred = y_pred.argmax(dim = 1, keepdim = True)
        correct += pred.eq(y.view_as(pred)).sum().item()
acc = np.round((100. * correct / len(val_data)), 4)
print(f'Accuracy: {acc}')

Accuracy: 47.7055


In [79]:
image = Image.open(r'C:\Users\Куликов\Desktop\gt.jpg').convert('RGB')
image_t = val_transform(image).to(device).unsqueeze(0)
resnet50.eval()
with torch.no_grad():
    y = resnet50(image_t)
    y_pred = y.argmax(dim = 1, keepdim = True)

In [80]:
y_pred

tensor([[114]], device='cuda:0')